In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q scarches SEACells faiss-gpu-cu12 scib-metrics

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.2/174.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 641.1/641.1 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.7/293.7 kB 30.3 MB/s

In [ ]:
import anndata
if not hasattr(anndata, "read"):
    anndata.read = anndata.read_h5ad
import os
os.environ["SCIPY_ARRAY_API"] = "1"
# Set paths for Google Colab
os.environ['HOME_DIR'] = '/content/drive/MyDrive/'
os.environ['MODEL_DIR'] = '/content/drive/MyDrive/models/'
os.environ['DATA_DIR'] = '/content/drive/MyDrive/data/'
os.environ['CODE_DIR'] = '/content/drive/MyDrive/codes/interpretable-prototype/'
os.environ['ISLANDER_SRC'] = '/content/drive/MyDrive/codes/Islander/src'
os.chdir('/content/drive/MyDrive/codes/interpretable-prototype/')
from importlib import reload
import interpretable_ssl.configs.larc
reload(interpretable_ssl.configs.larc)
import interpretable_ssl.trainers.scproto
reload(interpretable_ssl.trainers.scproto)
from interpretable_ssl.trainers.scproto import *

# generate gt context

## calc context + plot umap

In [88]:
t = SCProtoTrainer(dataset_id='s28f', debug=1, workers=0, lambda_swav = 1, lambda_kl=0, cvae_epochs = 5, pretraining_epochs = 10, affinity_type='gt')
def build_context(ad, radius):
    Xsp = ad.obsm["spatial"]
    Xpca = ad.obsm["X_pca"]
    nn_sp = NearestNeighbors(radius=radius).fit(Xsp)
    neigh = nn_sp.radius_neighbors(Xsp, return_distance=False)
    neigh = [idx[idx != i] for i, idx in enumerate(neigh)]
    ctx = np.stack([Xpca[idx].mean(0) if len(idx) else Xpca[i] for i, idx in enumerate(neigh)])
    return ctx
def save_aff(W, ad, ds_id, aff_type):
  save_path = f"./graphs/affinity_{ds_id}{len(ad)}_ncomp50_kneighbors50_{aff_type}.pkl"
  pkl.dump(W, open(save_path, "wb"))

ad = t.dataset.adata
ad.obsm['context'] = build_context(ad, 10)

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.


## gt affinity + umap

In [99]:
import numpy as np
import scanpy as sc

X = ad.obsm["context"].astype(np.float32)
codes, _ = pd.factorize(ad.obs["niches_2D"].values)   # 0..K-1

# pick a big margin compared to typical context distances
# (rule of thumb: 100x the median norm)
M = 10.0 * np.median(np.linalg.norm(X, axis=1))

X_aug = np.concatenate([X, (codes[:, None].astype(np.float32) * M)], axis=1)
ad.obsm["context_nichelocked"] = X_aug

sc.pp.neighbors(ad, use_rep="context_nichelocked", n_neighbors=50, method="umap")
sc.tl.umap(ad)

sc.pl.umap(ad, color= 'niches_2D')

In [100]:
W = ad.obsp["connectivities"]
effk = np.asarray(W.sum(axis=1)).ravel()
print(effk.min(), np.median(effk), effk.max())

5.6438446 8.72531 38.120583


In [101]:
save_aff(W, ad, 's28f', 'ugt')

# check if any signal exist in pca

In [22]:
ad = t.dataset.adata

In [23]:
# Check if niches are separable in gene/PCA space
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
X = ad.obsm['X_pca']
y = ad.obs['niches_2D']

# KNN accuracy on PCA
scores = cross_val_score(KNeighborsClassifier(15), X, y, cv=5)
print(f"Niche KNN accuracy (PCA): {scores.mean():.1%}")

# Baseline: random
from collections import Counter
majority = Counter(y).most_common(1)[0][1] / len(y)
print(f"Majority baseline: {majority:.1%}")

Niche KNN accuracy (PCA): 43.6%
Majority baseline: 38.3%


# run 1: no sinkhorn, initial lambda recon [0.5, 0.5]

In [24]:
# === RELOAD ===
import importlib
import interpretable_ssl.configs.defaults as defaults
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(defaults)
importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer

# === TRAIN ===
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # Affinity
    affinity_type='ugt',  # your niche-aware affinity

    # Views
    nmb_views=5,

    # Assignment metric
    assignment_metric='sneuc',

    # Temperature - let it auto-calibrate or set manually
    auto_eps_tau=1,
    # epsilon=0.5,
    # temperature=1.0,

    # Loss weights - reduce reconstruction, boost SwAV
    lambda_swav=1.0,
    lambda_recon=0.5,
    lambda_proto_recon=0.5,
    lambda_kl=0,
    lambda_aff=0,
    lambda_r1r2=0,

    # Epochs
    cvae_epochs=5,
    pretraining_epochs=20,

    # Other
    p=0,
    sinkhorn_iterations=0,
)

t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 19/20 | Loss: 1083.2216 | niche_mi: 0.439 | niche_Ma: 0.518 | unused: 4.0% | KNN: 39.1% (pca:43.3%)


Epoch 19:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 20/20 | Loss: 1083.1330 | niche_mi: 0.439 | niche_Ma: 0.512 | unused: 4.5% | KNN: 39.0% (pca:43.3%)


In [27]:
import importlib
import interpretable_ssl.trainers.scproto as sm
importlib.reload(sm)
t.niche_report = sm.SCProtoTrainer.niche_report.__get__(t, type(t))
t.niche_report()

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

PER-NICHE REPORT (sorted by KNN improvement)
Niche                          N #Proto  Purity  Cover  KNN_pca   KNN_z   Delta
--------------------------------------------------------------------------------
Macrophage islands          1047      0    0.0%   0.0%     7.2%    6.3%   -0.9%
Airways                      363      0    0.0%   0.0%    19.0%   17.9%   -1.1%
Desmoplastic stroma         5856    140   48.4% 176.5%    87.0%   83.1%   -3.9%
Tumor surface               3147     28   42.5% 117.2%    51.9%   47.9%   -4.0%
Alveolar spaces              415      1  100.0%   0.2%    14.7%   10.4%   -4.3%
Smooth muscle structures     282      2   30.7%  10.3%    12.4%    6.4%   -6.0%
Tumor core                   494      2   71.4%   1.6%     8.1%    2.0%   -6.1%
T cell aggregates           1001      4   25.3%  77.2%    29.7%   23.3%   -6.4%
Vascular stroma             2179      6   27.2%  21.8%    27.5%   11.4%  -16.2%
--------------------------------------------------------------------------

,niche,n_cells,n_protos,purity,coverage,knn_pca,knn_z,knn_delta
3,Macrophage islands,1047,0,0.000000,0.000000,0.071633,0.063037,-0.008596
0,Airways,363,0,0.000000,0.000000,0.190083,0.179063,-0.011019
2,Desmoplastic stroma,5856,140,0.484370,1.764857,0.870048,0.831113,-0.038934
7,Tumor surface,3147,28,0.425465,1.171592,0.518907,0.479187,-0.039720
1,Alveolar spaces,415,1,1.000000,0.002410,0.146988,0.103614,-0.043373
4,Smooth muscle structures,282,2,0.306548,0.102837,0.124113,0.063830,-0.060284
6,Tumor core,494,2,0.714286,0.016194,0.080972,0.020243,-0.060729
5,T cell aggregates,1001,4,0.252952,0.772228,0.296703,0.232767,-0.063936
8,Vascular stroma,2179,6,0.271585,0.218449,0.275356,0.113814,-0.161542


In [29]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D', show_proto_nums=False)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

r1: v0

In [121]:
# === RELOAD ===
import importlib
import interpretable_ssl.configs.defaults as defaults
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(defaults)
importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer

# === TRAIN ===
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # Affinity
    affinity_type='ugt',  # your niche-aware affinity

    # Views
    nmb_views=5,

    # Assignment metric
    assignment_metric='sneuc',

    # Temperature - let it auto-calibrate or set manually
    auto_eps_tau=1,
    # epsilon=0.5,
    # temperature=1.0,

    # Loss weights - reduce reconstruction, boost SwAV
    lambda_swav=1.0,
    lambda_recon=0.5,
    lambda_proto_recon=0.5,
    lambda_kl=0,
    lambda_aff=0,
    lambda_r1r2=0,

    # Epochs
    cvae_epochs=5,
    pretraining_epochs=20,

    # Other
    p=0,
    sinkhorn_iterations=0,
)

t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 19/20 | Loss: 1083.2209 | niche_mi: 0.439 | niche_Ma: 0.438 | unused: 4.0% | KNN: 39.2% (pca:43.3%)


Epoch 19:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 20/20 | Loss: 1083.1312 | niche_mi: 0.439 | niche_Ma: 0.509 | unused: 4.5% | KNN: 39.1% (pca:43.3%)


In [ ]:
niche 1: m1:0.3, m2:0.7
niche 2: m3:0.1 , m4:0.2
niche 3: m5:0.9

micro: avg(0.3, 0.7, 0.1, 0.2, 0.9)
macro: avg(avg(0.3, 0.7), avg(0.1, 0.2), 0.9)

In [26]:
import importlib
import interpretable_ssl.trainers.scproto as sm
importlib.reload(sm)
from interpretable_ssl.trainers.scproto import SCProtoTrainer

# Or bind to existing t:
t.niche_report = sm.SCProtoTrainer.niche_report.__get__(t, type(t))
t._niche_knn_acc = sm.SCProtoTrainer._niche_knn_acc.__get__(t, type(t))
t.niche_report()

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

PER-NICHE REPORT (sorted by KNN improvement)
Niche                          N #Proto  Purity  Cover  KNN_pca   KNN_z   Delta
--------------------------------------------------------------------------------
Macrophage islands          1047      0    0.0%   0.0%     7.2%    6.3%   -0.9%
Airways                      363      0    0.0%   0.0%    19.0%   17.9%   -1.1%
Desmoplastic stroma         5856    140   48.4% 176.5%    87.0%   83.1%   -3.9%
Tumor surface               3147     28   42.5% 117.2%    51.9%   47.9%   -4.0%
Alveolar spaces              415      1  100.0%   0.2%    14.7%   10.4%   -4.3%
Smooth muscle structures     282      2   30.7%  10.3%    12.4%    6.4%   -6.0%
Tumor core                   494      2   71.4%   1.6%     8.1%    2.0%   -6.1%
T cell aggregates           1001      4   25.3%  77.2%    29.7%   23.3%   -6.4%
Vascular stroma             2179      6   27.2%  21.8%    27.5%   11.4%  -16.2%
--------------------------------------------------------------------------

,niche,n_cells,n_protos,purity,coverage,knn_pca,knn_z,knn_delta
3,Macrophage islands,1047,0,0.000000,0.000000,0.071633,0.063037,-0.008596
0,Airways,363,0,0.000000,0.000000,0.190083,0.179063,-0.011019
2,Desmoplastic stroma,5856,140,0.484370,1.764857,0.870048,0.831113,-0.038934
7,Tumor surface,3147,28,0.425465,1.171592,0.518907,0.479187,-0.039720
1,Alveolar spaces,415,1,1.000000,0.002410,0.146988,0.103614,-0.043373
4,Smooth muscle structures,282,2,0.306548,0.102837,0.124113,0.063830,-0.060284
6,Tumor core,494,2,0.714286,0.016194,0.080972,0.020243,-0.060729
5,T cell aggregates,1001,4,0.252952,0.772228,0.296703,0.232767,-0.063936
8,Vascular stroma,2179,6,0.271585,0.218449,0.275356,0.113814,-0.161542


In [123]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# run 0: train using ugt

In [103]:
import importlib

# Reload all modified modules
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # More neighbors for stronger signal
    nmb_views=5,  # 4 neighbors + 1 anchor

    # Sharper assignments
    epsilon=0.8,
    temperature=1.25,
    # Use distance-based assignment instead of dot product
    assignment_metric='sneuc',  # negative euclidean

    # Keep more neighbors from affinity
    p=0,  # or p=0 to disable filtering
    lambda_kl=0,
    affinity_type='ugt',
    # More prototypes for granularity
    sinkhorn_iterations=0,
    lambda_swav=1,
    lambda_recon = 0.5,
    lambda_proto_recon = 0.5,
    lambda_r1r2=0.0,
    lambda_aff = 0,
    pretraining_epochs = 10,
    cvae_epochs = 5,
    auto_eps_tau=1

)
t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 9/10 | Loss: 1089.2831 | niche_micro: 0.413 | niche_macro: 0.366 | Proto unused: 0.00%


Epoch 9:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 10/10 | Loss: 1088.8475 | niche_micro: 0.414 | niche_macro: 0.348 | Proto unused: 0.00%


In [104]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [105]:
t.continue_training(20)

Epoch 10:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 11/30 | Loss: 1088.6599 | niche_micro: 0.413 | niche_macro: 0.347 | Proto unused: 0.00%


Epoch 11:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 29/30 | Loss: 1087.6223 | niche_micro: 0.415 | niche_macro: 0.456 | Proto unused: 0.50%


Epoch 29:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 30/30 | Loss: 1087.5351 | niche_micro: 0.415 | niche_macro: 0.458 | Proto unused: 0.00%


{'loss': 1087.535069878217,
 'batch_time': 0.608672293027242,
 'data_time': 0.5842466672261556,
 'swav': 5.731214843356046,
 'recon': 1083.1143674277973,
 'proto_recon': 1080.4933625852364,
 'kl': 12.18257862881962,
 'kl_balance': 12.18257862881962,
 'proto': 1.257084278482889,
 'commit': 0.9015355439397588,
 'p_matched': 0.015236135606505977,
 'q_matched': 0.015236135606505977,
 'z_norm': 3.3599983719205224,
 'proto_norm': 3.3905394077301025,
 'pproto_utilization': 0.9638960742047162,
 'qproto_utilization': 0.795183225553596,
 'p_uncertainty': 4.562059312480677,
 'q_uncertainty': 3.89199808319947,
 'compactness': 0.0,
 'separation': 0.0,
 'proto_entropy': 5.21164571294374,
 'aff': 0.0,
 'q_effk': 54.0406713807783,
 'p_effk': 101.57996598351546,
 'uniform': -5.180634498035536,
 'r1r2': 0.0,
 'proto_unused': 0.0,
 'proto_utilization': 0.9570512175559998,
 'lambda_kl': 0,
 'lambda_recon': 0.5,
 'niche_micro': np.float64(0.4152459337644523),
 'niche_macro': np.float64(0.45776320701267864)

In [106]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# sinkhorn - no recon

In [6]:
import importlib

# Reload all modified modules
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # More neighbors for stronger signal
    nmb_views=5,  # 4 neighbors + 1 anchor

    # Use distance-based assignment instead of dot product
    assignment_metric='sneuc',  # negative euclidean

    # Keep more neighbors from affinity
    p=0,  # or p=0 to disable filtering
    lambda_kl=0,
    affinity_type='ugt',
    # More prototypes for granularity
    sinkhorn_iterations=3,
    lambda_swav=1,
    lambda_recon = 0.0,
    lambda_proto_recon = 0.0,
    lambda_r1r2=0.0,
    lambda_aff = 0,
    pretraining_epochs = 30,
    cvae_epochs = 5,
    auto_eps_tau=1

)
t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 29/30 | Loss: 5.5049 | niche_mi: 0.405 | niche_Ma: 0.559 | unused: 73.0% | KNN: 39.7% (pca:43.3%)


Epoch 29:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 30/30 | Loss: 5.5015 | niche_mi: 0.405 | niche_Ma: 0.563 | unused: 71.0% | KNN: 39.7% (pca:43.3%)


In [7]:
t.niche_report()

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

PER-NICHE REPORT (sorted by KNN improvement)
Niche                          N  Purity  Cover  KNN_pca   KNN_z   Delta
----------------------------------------------------------------------
Airways                      363   10.3%  43.3%    19.0%   22.0%   +3.0%
Macrophage islands          1047    8.9%  58.7%     7.2%    5.3%   -1.9%
Alveolar spaces              415    2.5%  41.2%    14.7%   12.8%   -1.9%
Smooth muscle structures     282    7.8%  42.2%    12.4%    9.6%   -2.8%
Desmoplastic stroma         5856   39.9%  46.9%    87.0%   83.4%   -3.6%
Tumor surface               3147   34.9%  48.6%    51.9%   48.0%   -3.8%
T cell aggregates           1001    9.4%  65.0%    29.7%   23.8%   -5.9%
Tumor core                   494    6.5%  57.3%     8.1%    1.4%   -6.7%
Vascular stroma             2179   15.7%  49.5%    27.5%   14.2%  -13.3%
----------------------------------------------------------------------
MEAN                               15.1%  50.3%    28.6%   24.5%   -4.1%


,niche,n_cells,purity,coverage,knn_pca,knn_z,knn_delta
0,Airways,363,0.102951,0.432507,0.190083,0.220386,0.030303
3,Macrophage islands,1047,0.089247,0.587393,0.071633,0.052531,-0.019102
1,Alveolar spaces,415,0.024815,0.412048,0.146988,0.127711,-0.019277
4,Smooth muscle structures,282,0.078033,0.421986,0.124113,0.095745,-0.028369
2,Desmoplastic stroma,5856,0.398636,0.469092,0.870048,0.834358,-0.035690
7,Tumor surface,3147,0.348620,0.485542,0.518907,0.480458,-0.038449
5,T cell aggregates,1001,0.094471,0.650350,0.296703,0.237762,-0.058941
6,Tumor core,494,0.064568,0.572874,0.080972,0.014170,-0.066802
8,Vascular stroma,2179,0.156581,0.495181,0.275356,0.142267,-0.133089


In [12]:
len(t.dataset)

15309

In [18]:
  import importlib
  import interpretable_ssl.trainers.trainer as trainer_module
  importlib.reload(trainer_module)
  t.plot_umap_simple = trainer_module.Trainer.plot_umap_simple.__get__(t)

  fig, proto_labels = t.plot_umap_simple(color_key='niches_2D', figsize=(7, 4),
  show_proto_nums=False)

Output hidden; open in https://colab.research.google.com to view.

# swav 2, recon, proto recon 0.1, sinkhorn = 0 -> collapse

In [112]:
# === RELOAD ===
import importlib
import interpretable_ssl.configs.defaults as defaults
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(defaults)
importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer

# === TRAIN ===
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # Affinity
    affinity_type='ugt',  # your niche-aware affinity

    # Views
    nmb_views=5,

    # Assignment metric
    assignment_metric='sneuc',

    # Temperature - let it auto-calibrate or set manually
    auto_eps_tau=1,
    # epsilon=0.5,
    # temperature=1.0,

    # Loss weights - reduce reconstruction, boost SwAV
    lambda_swav=2.0,
    lambda_recon=0.1,
    lambda_proto_recon=0.1,
    lambda_kl=0,
    lambda_aff=0,
    lambda_r1r2=0,

    # Epochs
    cvae_epochs=5,
    pretraining_epochs=20,

    # Other
    p=0,
    sinkhorn_iterations=0,
)

t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 19/20 | Loss: 226.3920 | niche_mi: 0.398 | niche_Ma: 0.604


Epoch 19:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 20/20 | Loss: 226.3977 | niche_mi: 0.398 | niche_Ma: 0.606 | KNN: 38.9% (pca:43.3%)


In [114]:
import importlib
import interpretable_ssl.trainers.scproto as sm
importlib.reload(sm)
from interpretable_ssl.trainers.scproto import SCProtoTrainer

# Or bind to existing t:
t.niche_report = sm.SCProtoTrainer.niche_report.__get__(t, type(t))
t._niche_knn_acc = sm.SCProtoTrainer._niche_knn_acc.__get__(t, type(t))
t.niche_report()


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

PER-NICHE REPORT (sorted by KNN improvement)
Niche                          N  Purity  Cover  KNN_pca   KNN_z   Delta
----------------------------------------------------------------------
Macrophage islands          1047    7.7%  45.8%     7.2%    5.7%   -1.4%
Airways                      363    3.2%  49.6%    19.0%   15.4%   -3.6%
Desmoplastic stroma         5856   38.6%  41.0%    87.0%   83.4%   -3.6%
Tumor surface               3147   19.4%  38.4%    51.9%   47.2%   -4.7%
Alveolar spaces              415    3.6%  48.7%    14.7%    9.6%   -5.1%
Tumor core                   494    2.7%  34.0%     8.1%    2.6%   -5.5%
T cell aggregates           1001    9.4%  58.5%    29.7%   23.6%   -6.1%
Smooth muscle structures     282    3.0%  58.2%    12.4%    5.7%   -6.7%
Vascular stroma             2179   16.0%  40.7%    27.5%   10.5%  -17.1%
----------------------------------------------------------------------
MEAN                               11.5%  46.1%    28.6%   22.6%   -6.0%


,niche,n_cells,purity,coverage,knn_pca,knn_z,knn_delta
3,Macrophage islands,1047,0.077059,0.458453,0.071633,0.057307,-0.014327
0,Airways,363,0.032485,0.495868,0.190083,0.154270,-0.035813
2,Desmoplastic stroma,5856,0.385616,0.410178,0.870048,0.834016,-0.036031
7,Tumor surface,3147,0.193932,0.383858,0.518907,0.471560,-0.047347
1,Alveolar spaces,415,0.036456,0.486747,0.146988,0.096386,-0.050602
6,Tumor core,494,0.026971,0.340081,0.080972,0.026316,-0.054656
5,T cell aggregates,1001,0.094076,0.585415,0.296703,0.235764,-0.060939
4,Smooth muscle structures,282,0.029598,0.581560,0.124113,0.056738,-0.067376
8,Vascular stroma,2179,0.160079,0.407067,0.275356,0.104635,-0.170721


In [115]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## we have collpase here, why? recon and proto recon very low

# swav 2, recon, proto recn = 0.5 -> slight collapse

In [117]:
# === RELOAD ===
import importlib
import interpretable_ssl.configs.defaults as defaults
import interpretable_ssl.trainers.trainer as tm
import interpretable_ssl.trainers.adaptive_trainer as at
import interpretable_ssl.trainers.scproto as sm

importlib.reload(defaults)
importlib.reload(tm)
importlib.reload(at)
importlib.reload(sm)

from interpretable_ssl.trainers.scproto import SCProtoTrainer

# === TRAIN ===
t = SCProtoTrainer(
    dataset_id='s28f',
    debug=1,
    workers=0,

    # Affinity
    affinity_type='ugt',  # your niche-aware affinity

    # Views
    nmb_views=5,

    # Assignment metric
    assignment_metric='sneuc',

    # Temperature - let it auto-calibrate or set manually
    auto_eps_tau=1,
    # epsilon=0.5,
    # temperature=1.0,

    # Loss weights - reduce reconstruction, boost SwAV
    lambda_swav=2.0,
    lambda_recon=0.5,
    lambda_proto_recon=0.5,
    lambda_kl=0,
    lambda_aff=0,
    lambda_r1r2=0,

    # Epochs
    cvae_epochs=5,
    pretraining_epochs=20,

    # Other
    p=0,
    sinkhorn_iterations=3,
)

t.setup()
t.run()

dataset is None, loading s28f
loading s28f data
⚠️ No HVG column found.
Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 960 31 10
	Mean/Var Layer in/out: 31 8
Decoder Architecture:
	First Layer in, out and cond:  8 31 10
	Output Layer in/out:  31 960 



INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


📊 Affinity: mean_deg=76.4, effk_med=37.0, mutual=100.00%
adam
The model is being trained without using prototypes.
Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 1085.20 - val_cvae_loss: 1085.20


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

>>> Epoch 19/20 | Loss: 1090.0108 | niche_mi: 0.429 | niche_Ma: 0.383 | unused: 6.0% | KNN: 39.0% (pca:43.3%)


Epoch 19:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

>>> Epoch 20/20 | Loss: 1089.9933 | niche_mi: 0.429 | niche_Ma: 0.406 | unused: 5.5% | KNN: 39.1% (pca:43.3%)


In [118]:
import importlib
import interpretable_ssl.trainers.scproto as sm
importlib.reload(sm)
from interpretable_ssl.trainers.scproto import SCProtoTrainer

# Or bind to existing t:
t.niche_report = sm.SCProtoTrainer.niche_report.__get__(t, type(t))
t._niche_knn_acc = sm.SCProtoTrainer._niche_knn_acc.__get__(t, type(t))
t.niche_report()

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

PER-NICHE REPORT (sorted by KNN improvement)
Niche                          N  Purity  Cover  KNN_pca   KNN_z   Delta
----------------------------------------------------------------------
Airways                      363   15.0%   9.4%    19.0%   19.8%   +0.8%
Macrophage islands          1047   13.0%   5.1%     7.2%    6.3%   -0.9%
Tumor surface               3147   46.4%   5.7%    51.9%   48.9%   -3.0%
Desmoplastic stroma         5856   45.0%   2.8%    87.0%   83.4%   -3.6%
Tumor core                   494   12.2%   9.5%     8.1%    3.8%   -4.3%
Smooth muscle structures     282   10.2%   9.2%    12.4%    7.4%   -5.0%
T cell aggregates           1001   25.0%  10.2%    29.7%   23.7%   -6.0%
Alveolar spaces              415   10.2%   6.3%    14.7%    8.2%   -6.5%
Vascular stroma             2179   18.9%   2.8%    27.5%   11.8%  -15.7%
----------------------------------------------------------------------
MEAN                               21.8%   6.8%    28.6%   23.7%   -4.9%


,niche,n_cells,purity,coverage,knn_pca,knn_z,knn_delta
0,Airways,363,0.149780,0.093664,0.190083,0.198347,0.008264
3,Macrophage islands,1047,0.129902,0.050621,0.071633,0.063037,-0.008596
7,Tumor surface,3147,0.463731,0.056880,0.518907,0.488719,-0.030187
2,Desmoplastic stroma,5856,0.450000,0.027664,0.870048,0.834358,-0.035690
6,Tumor core,494,0.121762,0.095142,0.080972,0.038462,-0.042510
4,Smooth muscle structures,282,0.102362,0.092199,0.124113,0.074468,-0.049645
5,T cell aggregates,1001,0.250000,0.101898,0.296703,0.236763,-0.059940
1,Alveolar spaces,415,0.102362,0.062651,0.146988,0.081928,-0.065060
8,Vascular stroma,2179,0.188854,0.027994,0.275356,0.117944,-0.157412


In [119]:
fig, proto_labels = t.plot_umap_simple(color_key= 'niches_2D')
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [120]:
t.sinkhorn_iterations

3